****

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readcsv").getOrCreate()

In [2]:
!pip install faker
import os
import random
from faker import Faker
import pandas as pd
from datetime import datetime, timedelta

fake = Faker("en_IN")
random.seed(42)

OUTPUT_DIR = "Azure_DataEngineering_Datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------
# Configuration
# ------------------------------------------

NUM_DEPARTMENTS = 20
NUM_EMPLOYEES = 1000
NUM_CUSTOMERS = 5000
NUM_PRODUCTS = 5000
NUM_ORDERS = 5000
NUM_SALES = 10000
NUM_INVENTORY = 1000
NUM_SALARY_HISTORY = 1000

# ------------------------------------------
# Departments
# ------------------------------------------

departments = [
    "IT","HR","Finance","Sales","Marketing",
    "Operations","Support","Legal","Admin",
    "Procurement","Analytics","Engineering",
    "Security","Cloud","DevOps","QA",
    "Business","Compliance","Healthcare","Retail"
]

dept_df = pd.DataFrame({
    "DeptID": range(1, NUM_DEPARTMENTS+1),
    "DepartmentName": departments
})

dept_df.to_csv(f"{OUTPUT_DIR}/Department.csv", index=False)

print("Department.csv created")

# ------------------------------------------
# Employees
# ------------------------------------------

employees = []

manager_ids = []

for emp_id in range(1, NUM_EMPLOYEES+1):

    dept = random.randint(1, NUM_DEPARTMENTS)

    join_date = fake.date_between(
        start_date='-10y',
        end_date='today'
    )

    salary = random.randint(3000,25000)

    if emp_id <= 100:
        manager = None
        manager_ids.append(emp_id)
    else:
        manager = random.choice(manager_ids)

    employees.append([
        emp_id,
        fake.name(),
        dept,
        manager,
        salary,
        join_date,
        fake.city(),
        fake.state(),
        fake.email(),
        fake.phone_number()
    ])

emp_df = pd.DataFrame(employees, columns=[
    "EmpID","EmployeeName","DeptID",
    "ManagerID","Salary","JoinDate",
    "City","State","Email","Phone"
])

emp_df.to_csv(f"{OUTPUT_DIR}/Employee.csv", index=False)

print("Employee.csv created")

# ------------------------------------------
# Employee Manager
# ------------------------------------------

manager_df = emp_df[["EmpID","EmployeeName","ManagerID"]].copy()

manager_df["ManagerName"] = manager_df["ManagerID"].apply(
    lambda x:
    emp_df.loc[emp_df.EmpID==x,"EmployeeName"].values[0]
    if pd.notna(x) else None
)

manager_df.to_csv(
    f"{OUTPUT_DIR}/Employee_Manager.csv",
    index=False
)

print("Employee_Manager.csv created")

# ------------------------------------------
# Customers
# ------------------------------------------

customers=[]

for cid in range(1,NUM_CUSTOMERS+1):

    customers.append([
        cid,
        fake.name(),
        fake.city(),
        fake.state(),
        fake.country(),
        fake.email(),
        fake.phone_number(),
        fake.date_between('-8y','today')
    ])

cust_df=pd.DataFrame(customers,columns=[
    "CustomerID",
    "CustomerName",
    "City",
    "State",
    "Country",
    "Email",
    "Phone",
    "RegistrationDate"
])

cust_df.to_csv(
    f"{OUTPUT_DIR}/Customer.csv",
    index=False
)

print("Customer.csv created")

# ------------------------------------------
# Products
# ------------------------------------------

categories=[
"Electronics","Furniture","Fashion",
"Grocery","Books","Healthcare",
"Sports","Home","Beauty","Automotive"
]

products=[]

for pid in range(1,NUM_PRODUCTS+1):

    price=random.randint(100,50000)

    products.append([
        pid,
        fake.word().capitalize()+" Product",
        random.choice(categories),
        price
    ])

prod_df=pd.DataFrame(products,columns=[
"ProductID","ProductName","Category","Price"
])

prod_df.to_csv(
    f"{OUTPUT_DIR}/Product.csv",
    index=False
)

print("Product.csv created")

# ------------------------------------------
# Orders
# ------------------------------------------

orders=[]

for oid in range(1,NUM_ORDERS+1):

    orders.append([
        oid,
        random.randint(1,NUM_CUSTOMERS),
        fake.date_between('-5y','today'),
        random.choice([
            "Completed",
            "Cancelled",
            "Pending",
            "Returned"
        ])
    ])

order_df=pd.DataFrame(
orders,
columns=[
"OrderID",
"CustomerID",
"OrderDate",
"OrderStatus"
])

order_df.to_csv(
f"{OUTPUT_DIR}/Orders.csv",
index=False
)

print("Orders.csv created")

# ------------------------------------------
# Sales
# ------------------------------------------

chunk_size=100000

for chunk in range(NUM_SALES//chunk_size):

    sales=[]

    for _ in range(chunk_size):

        qty=random.randint(1,10)

        price=random.randint(100,50000)

        sales.append([
            random.randint(1,NUM_ORDERS),
            random.randint(1,NUM_PRODUCTS),
            qty,
            price,
            qty*price,
            fake.date_between('-5y','today')
        ])

    pd.DataFrame(
    sales,
    columns=[
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "TotalAmount",
    "SaleDate"
    ]).to_csv(
    f"{OUTPUT_DIR}/Sales.csv",
    mode='a',
    index=False,
    header=(chunk==0)
    )

    print(f"Sales Chunk {chunk+1} completed")

print("Sales.csv created")

# ------------------------------------------
# Inventory
# ------------------------------------------

inventory=[]

for _ in range(NUM_INVENTORY):

    inventory.append([
        random.randint(1,NUM_PRODUCTS),
        fake.city(),
        random.randint(0,5000),
        fake.date_between('-2y','today')
    ])

inventory_df=pd.DataFrame(
inventory,
columns=[
"ProductID",
"Warehouse",
"StockQty",
"LastUpdated"
])

inventory_df.to_csv(
f"{OUTPUT_DIR}/Inventory.csv",
index=False
)

print("Inventory.csv created")

# ------------------------------------------
# Salary History
# ------------------------------------------

history=[]

for _ in range(NUM_SALARY_HISTORY):

    emp=random.randint(1,NUM_EMPLOYEES)

    start=fake.date_between('-8y','-2y')

    end=start+timedelta(days=random.randint(200,700))

    history.append([
        emp,
        random.randint(30000,250000),
        start,
        end
    ])

salary_df=pd.DataFrame(
history,
columns=[
"EmpID",
"Salary",
"EffectiveFrom",
"EffectiveTo"
])

salary_df.to_csv(
f"{OUTPUT_DIR}/Salary_History.csv",
index=False
)

print("Salary_History.csv created")

print("\n================================")
print("All CSV Files Generated")
print("================================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.4 MB/s eta 0:00:00
Department.csv created
Employee.csv created
Employee_Manager.csv created
Customer.csv created
Product.csv created
Orders.csv created
Sales.csv created
Inventory.csv created
Salary_History.csv created

All CSV Files Generated


The datasets are currently being saved to the `Azure_DataEngineering_Datasets` folder, which is created in the same directory where this Jupyter notebook is running. This is generally considered your 'Jupyter directory'.

If you wish to change the output location, you can modify the `OUTPUT_DIR` variable in the code block above.

For example:
- To save files directly in the notebook's root directory, you can set `OUTPUT_DIR = '.'` or `OUTPUT_DIR = ''`.
- To save files in a different folder, like `my_data_exports`, set `OUTPUT_DIR = 'my_data_exports'`.

In [3]:
import os

# Assuming OUTPUT_DIR is defined from previous cells
# If not, you might need to define it: OUTPUT_DIR = "Azure_DataEngineering_Datasets"

print(f"Files generated in: {OUTPUT_DIR}")
print("----------------------------------")
for filename in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, filename)
    print(file_path)

Files generated in: Azure_DataEngineering_Datasets
----------------------------------
Azure_DataEngineering_Datasets/Department.csv
Azure_DataEngineering_Datasets/Employee.csv
Azure_DataEngineering_Datasets/Customer.csv
Azure_DataEngineering_Datasets/Product.csv
Azure_DataEngineering_Datasets/Inventory.csv
Azure_DataEngineering_Datasets/Orders.csv
Azure_DataEngineering_Datasets/Salary_History.csv
Azure_DataEngineering_Datasets/Employee_Manager.csv


In [10]:
df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("Azure_DataEngineering_Datasets/Employee.csv")
)
#df.createOrReplaceTempView('Employee')
# df1 = df.filter(df.ManagerID.isNull())\
#         .withColumn('Level', lit(1))\
#         .select('EmpID','EmployeeName','ManagerID','Level')
# df2 = df.join(df, df.ManagerID == df.EmpID, 'inner').select()
spark.sql(f"""WITH EmployeeHierarchy AS
(SELECT
        EmpID,
        EmployeeName,
        ManagerID,
        1 AS Level
    FROM Employee
    WHERE ManagerID IS NULL

    UNION ALL

    SELECT
        e.EmpID,
        e.EmployeeName,
        e.ManagerID,
        eh.Level + 1
    FROM Employee e
    INNER JOIN Employee eh
        ON e.ManagerID = eh.EmpID
)

SELECT *
FROM EmployeeHierarchy""").show(20)

{"ts": "2026-07-08 17:25:58.708", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `eh`.`Level` cannot be resolved. Did you mean one of the following? [`e`.`City`, `eh`.`City`, `e`.`Email`, `eh`.`Email`, `e`.`Phone`]. SQLSTATE: 42703", "context": {"errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o27.sql.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `eh`.`Level` cannot be resolved. Did you mean one of the following? [`e`.`City`, `eh`.`City`, `e`.`Email`, `eh`.`Email`, `e`.`Phone`]. SQLSTATE: 42703; line 16 pos 8;\n'Project [*]\n+- 'SubqueryAlias EmployeeHierarchy\n   +- 'SubqueryAlias EmployeeHierarchy\n      +- 'Union false, false\n         :- Project [EmpID#254, EmployeeName#255, ManagerID#257, 1 AS Level#302]\n         

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `eh`.`Level` cannot be resolved. Did you mean one of the following? [`e`.`City`, `eh`.`City`, `e`.`Email`, `eh`.`Email`, `e`.`Phone`]. SQLSTATE: 42703; line 16 pos 8;
'Project [*]
+- 'SubqueryAlias EmployeeHierarchy
   +- 'SubqueryAlias EmployeeHierarchy
      +- 'Union false, false
         :- Project [EmpID#254, EmployeeName#255, ManagerID#257, 1 AS Level#302]
         :  +- Filter isnull(ManagerID#257)
         :     +- SubqueryAlias employee
         :        +- View (`Employee`, [EmpID#254, EmployeeName#255, DeptID#256, ManagerID#257, Salary#258, JoinDate#259, City#260, State#261, Email#262, Phone#263L])
         :           +- Relation [EmpID#254,EmployeeName#255,DeptID#256,ManagerID#257,Salary#258,JoinDate#259,City#260,State#261,Email#262,Phone#263L] csv
         +- 'Project [EmpID#303, EmployeeName#304, ManagerID#306, unresolvedalias(('eh.Level + 1))]
            +- Join I